# Notebook 03 - Article Figures And Tables

This notebook exports article-ready tables, captions, methods text, and interpretation notes after the pipeline has produced final model outputs.

In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
PROJECT_ROOT = next(
    path for path in _candidates
    if (path / "pyproject.toml").exists() and (path / "src" / "us_gdp_regime").exists()
)
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [2]:
from pathlib import Path

import json
import pandas as pd

from us_gdp_regime.article_exports import write_article_assets
from us_gdp_regime.config import load_config
from us_gdp_regime.pipeline import fit_models, make_figures, prepare_data

config = load_config(Path("config/default.yaml"))
if not (config.paths.processed_dir / "us_gdp_series.csv").exists():
    prepare_data(config)
model_outputs = fit_models(config)
figure_outputs = make_figures(config)
model_outputs | figure_outputs

C:\Users\diogo\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


{'trend': WindowsPath('data/models/trend_regression.csv'),
 'segments': WindowsPath('data/models/regime_segments.csv'),
 'trend_figure': WindowsPath('figures/log_real_gdp_trend.png'),
 'regimes_figure': WindowsPath('figures/gdp_growth_regimes.png'),
 'annotated_regimes_figure': WindowsPath('figures/gdp_growth_regimes_annotated.png')}

## Load Final Model Outputs

In [3]:
trend = pd.read_csv(config.paths.models_dir / "trend_regression.csv")
regimes = pd.read_csv(config.paths.models_dir / "regime_segments.csv")
display(trend)
display(regimes)

,intercept,slope,r_squared,annualised_growth_rate
0,-38.967841,0.031076,0.987,3.156432


,segment_id,start_year,end_year,n_observations,mean_growth,long_run_mean,regime,sse
0,1,1921,1928,8,3.188583,2.968201,above_mean,151.247028
1,2,1929,1933,5,-5.776430,2.968201,below_mean,233.248676
2,3,1934,1944,11,8.096041,2.968201,above_mean,270.197092
3,4,1945,1949,5,-1.981110,2.968201,below_mean,96.136868
4,5,1950,2022,73,3.109303,2.968201,above_mean,413.479715


## Export Article Assets

In [4]:
asset_outputs = write_article_assets(regimes, trend, output_dir=Path("article_assets"))
asset_outputs

{'article_regime_table': WindowsPath('article_assets/regime_table.md'),
 'article_trend_summary': WindowsPath('article_assets/trend_summary.md'),
 'article_figure_captions': WindowsPath('article_assets/figure_captions.json'),
 'article_methods_box': WindowsPath('article_assets/methods_box.md')}

## Trend Regression Table

In [5]:
print(Path("article_assets/trend_summary.md").read_text(encoding="utf-8"))

| Intercept | Log trend slope | R-squared | Annualised trend growth (%) |
| --- | --- | --- | --- |
| -38.968 | 0.031 | 0.987 | 3.156 |



## Regime Table

In [6]:
print(Path("article_assets/regime_table.md").read_text(encoding="utf-8"))

| Segment | Start | End | Years | Mean growth (%) | Long-run mean (%) | Regime |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1921 | 1928 | 8 | 3.19 | 2.97 | above_mean |
| 2 | 1929 | 1933 | 5 | -5.78 | 2.97 | below_mean |
| 3 | 1934 | 1944 | 11 | 8.10 | 2.97 | above_mean |
| 4 | 1945 | 1949 | 5 | -1.98 | 2.97 | below_mean |
| 5 | 1950 | 2022 | 73 | 3.11 | 2.97 | above_mean |



## Figure Captions

In [7]:
captions = json.loads(Path("article_assets/figure_captions.json").read_text(encoding="utf-8"))
for figure, caption in captions.items():
    print(f"{figure}: {caption}")
    print()

fred_maddison_growth_comparison.png: Overlapping annual real GDP growth rates from Maddison-derived GDP and FRED/BEA GDPCA. The comparison is a validation diagnostic, not a causal test.

gdp_growth_regimes.png: Annual United States real GDP growth with piecewise constant growth regimes. Source: Maddison Project Database 2023; method: dynamic programming segmentation selected by the configured information criterion.

gdp_growth_regimes_annotated.png: Annual United States real GDP growth regimes with selected historical context. Source: Maddison Project Database 2023; annotations are descriptive context and are not identified causes of statistical breaks.

log_real_gdp_trend.png: United States real GDP on a log scale with a fitted linear trend. Source: Maddison Project Database 2023; method: ordinary least squares trend regression on log real GDP.



## Methods Box

In [8]:
print(Path("article_assets/methods_box.md").read_text(encoding="utf-8"))

Methods: The historical series uses Maddison Project Database 2023 GDP per capita and population to construct a United States real GDP proxy from 1920 onward. Annual growth rates are computed from that real GDP proxy. The trend model is a linear regression on log real GDP. Growth regimes are estimated with a piecewise constant mean model on annual growth rates, using dynamic programming and the configured information criterion. FRED/BEA GDPCA is used as an overlapping validation source from 1929 onward when available. Statistical breaks structure the description; they do not by themselves identify historical causes.



## Historical Interpretation Notes

Use statistical regimes as structure, then add historical context cautiously. Candidate context includes the Great Depression, World War II mobilisation and demobilisation, postwar expansion, the productivity slowdown after the early 1970s, Volcker disinflation, the Global Financial Crisis, and COVID-19. These events should be phrased as coincident context unless supported by additional historical evidence.

In [9]:
notes = pd.DataFrame({
    "period": [
        "Great Depression",
        "World War II mobilisation",
        "Postwar adjustment",
        "1970s productivity slowdown and oil shocks",
        "Global Financial Crisis",
        "COVID-19 shock",
    ],
    "interpretation_rule": [
        "Describe as context for unusually weak growth unless separately sourced.",
        "Treat mobilisation as historical context, not automatic model-implied causality.",
        "Separate statistical break evidence from demobilisation interpretation.",
        "Use language such as coincides with or is consistent with.",
        "Mention as a major macroeconomic disruption with citations.",
        "Treat the shock and rebound as exceptional annual movements.",
    ],
})
notes

,period,interpretation_rule
0,Great Depression,Describe as context for unusually weak growth ...
1,World War II mobilisation,"Treat mobilisation as historical context, not ..."
2,Postwar adjustment,Separate statistical break evidence from demob...
3,1970s productivity slowdown and oil shocks,Use language such as coincides with or is cons...
4,Global Financial Crisis,Mention as a major macroeconomic disruption wi...
5,COVID-19 shock,Treat the shock and rebound as exceptional ann...


## Rerun Note

This notebook is intended to be rerun after the data and modelling pipeline. Tables are deterministic and captions explicitly identify source and method.